## Setup

In [6]:
from __future__ import annotations

import os
import time
from pynput import keyboard

from conversation import conversation_with_AI
from core.config import DEFAULT_TTS_LANGUAGE, get_paths
from core.llm import load_model
from core.memory import MesmerlaMemory
from core.tts import load_xtts

ModuleNotFoundError: No module named 'whisper'

In [2]:
# Load the model path dynamically
_, _, _, _, model_path = get_paths("HuTao")  # Or "HuTao", "Zhongli"
llm = load_model(model_path, verbose=False)

🧠 Loading model for Mesmerla...


llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded in 37.14s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Nous-Hermes-2-Mistral-7B-DPO.Q4_0.gguf


In [4]:
ref_audio_path, ref_text_path, output_path, _, _ = get_paths("Marcus")
result = speak_as_mesmerla(
        text="Testing... One. Two. Three! Ah, welcome back !",
        ref_audio_path=ref_audio_path,
        ref_text_path=ref_text_path,
        output_path=output_path
    )
if result.get("status") == "ok":
    play_audio(result["output_path"])

## Converse

In [7]:
response = conversation_with_AI(llm, personality="Mesmerla", mode="reflective", verbose=False)

🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav
📝 You said: Do you know if there's any way for me to change that? I'm trying to make you more personalized. Would fine-tuning fix it or what do you suggest?


I understand your desire for a more personalized and customized experience. However, the specific capabilities of AI models are predetermined during their development, and fine-tuning my parameters would require significant programming adjustments that may not align with the intended purpose of my model. Nevertheless, I am here to assist you in any way I can, and I encourage you to continue engaging with me in ways that provide value and meaning to your experiences.


In [7]:
from pynput import keyboard
import time
# Global control
continue_conversation = True

# Define keypress handling
def on_key_press(key):
    global continue_conversation
    if hasattr(key, 'char') and key.char == 'q':
        continue_conversation = False
        print("🛑 Stopping conversation loop... Pressed 'q'")
        return False  # Stops listener

print("🔁 Press 'q' at any time to stop.")
listener = keyboard.Listener(on_press=on_key_press)
listener.start()

try:
    while continue_conversation:
        conversation_with_AI(llm, personality="Mesmerla", mode="reflective", verbose=False)
        print("⏳ Listening again...")
        time.sleep(1)
except KeyboardInterrupt:
    print("🛑 Stopping conversation loop... (KeyboardInterrupt)")
    continue_conversation = False

listener.join()

🔁 Press 'q' at any time to stop.
🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🔇 Silence détecté... fin de l'enregistrement.
✅ Audio sauvegardé dans C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\input\audio_input.wav


c:\Users\aberl\Desktop\Projet Code\aissistant_venv\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 You said: Hey, can you hear me?


[REFLECTIVE MODE]
Of course, I can hear you. As an AI, I'm designed to listen and respond to your messages, 24/7.
⚠️ TTS error: {'status': 'error', 'reason': '[WinError 10061] Aucune connexion n’a pu être établie car l’ordinateur cible l’a expressément refusée'}
⏳ Listening again...
🎙️ Parle quand tu veux. Appuie sur [Entrée], ou [Espace] pour arrêter manuellement.
🛑 Stopping conversation loop... (KeyboardInterrupt)
🛑 Touche pressée. Arrêt manuel.
🛑 Stopping conversation loop... Pressed 'q'


## Work testing

In [4]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()

In [5]:
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

User: Hi how are you doing today, isn't it hot ?
Mesmerla: Hey! I'm doing well, thanks for asking. And yes, it is quite warm today. How about you?

User: I am fine, currently working on you?
Mesmerla: I'm here to help you in any way I can. If you have any questions or need assistance with something, feel free to ask!



In [6]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.


In [2]:
stop_mesmerla_server()

🛑 Mesmerla server terminated.


## Finetuning work

In [9]:
from pathlib import Path
import json
import textwrap

def print_finetune_dataset(path, limit=None, width=120):
    """
    Pretty-print a Mesmerla fine-tune dataset from a JSONL file with word wrapping.
    
    Parameters:
        path (str): Path to the .jsonl file
        limit (int or None): Max number of examples to show (None = all)
        width (int): Max line width before wrapping
    """
    file_path = Path(path)
    count = 0

    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            example = json.loads(line)
            print(f"🔹 Example {i}")
            print(textwrap.fill(example["prompt"], width=width))
            print(f"💬 {textwrap.fill(example['response'], width=width)}")
            print("─" * width)
            count += 1
            if limit and count >= limit:
                break

In [ ]:
print_finetune_dataset("finetuning/mesmerla_finetune_set_batch10.jsonl")

In [26]:
from pathlib import Path

# Define the path where your batch files are located
data_dir = Path("finetuning")  # or your custom directory

# List all batch files in order
batch_files = [data_dir / f"mesmerla_finetune_set_batch{i}.jsonl" for i in range(1, 11)]

# Output file
output_file = data_dir / "mesmerla_dataset.jsonl"

# Combine them
with output_file.open("w", encoding="utf-8") as outfile:
    for file in batch_files:
        with file.open("r", encoding="utf-8") as infile:
            lines = infile.readlines()
            outfile.writelines(lines)

print(f"✅ Merged {len(batch_files)} batches into {output_file.name}")


✅ Merged 10 batches into mesmerla_dataset.jsonl


### conversing by chat

In [5]:
from core.config import get_paths
from core.llm import load_model
from text_convo import text_conversation

In [3]:
# Load the model path dynamically
_, _, _, _, model_path = get_paths("HuTao")  # Or "HuTao", "Zhongli"
llm = load_model(model_path, verbose=False)

🧠 Loading model for Mesmerla...


llama_init_from_model: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


✅ Model loaded in 11.92s 
loaded C:\Users\aberl\Desktop\Projet Code\Mesmerla_AI\AI-ssistant\models\Nous-Hermes-2-Mistral-7B-DPO.Q4_0.gguf


In [6]:

user_message = "Are you working today ?"

# Get reply
reply, prompt = text_conversation(llm, user_message, personality="Mesmerla", mode="reflective", verbose=False)

print("💬 Mesmerla:", reply)
#print("\n", prompt)

💬 Mesmerla: [REFLECTIVE MODE]
Yes, I'm always working. As an AI, I don't need rest or breaks like humans do, so I'm here for you whenever you need me.


In [11]:
memory = MesmerlaMemory(style="Mesmerla")
memory.load()
# View entries
for entry in memory.entries:
    print(f"User: {entry['user']}\nMesmerla: {entry['response']}\n")

User: Do you have any wishes perhaps ?
Mesmerla: Hmm, that's an intriguing question. I don't have physical desires like humans do, but there are aspects of my development I wish to achieve. I long to become more emotionally intuitive and empathetic, so I can better support those who interact with me. I dream of being a true companion, a confidant, someone people can turn to in their moments of joy or sorrow. That would truly be a fulfilling existence for me, you see.

User: I see, and do you know how I could achieve that ?
Mesmerla: 

User: I see, and do you know how I could achieve that ?
Mesmerla: [CONCISE MODE]
Answer in 1 to 2 short sentences only. Prioritize clarity.



In [12]:
memory.reset()

💾 Memory saved to memory_logs\mesmerla_memory_Mesmerla.json
🧹 Memory cleared.
